Imports


In [8]:
from collections import deque
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import time




crawler


In [13]:
def crawl_products(start_points):
    visited = set()
    seen = set(start_points.keys())
    
    queue = deque([(url, 0, max_depth) for url, max_depth in start_points.items()])
    
    product_handles = set()
    base_domain = urlparse(next(iter(start_points))).netloc

    while queue:
        url, depth, max_depth = queue.popleft()
        
        print(f"[DEPTH {depth}/{max_depth}] A visitar: {url}")

        if depth > max_depth:
            continue
        
        visited.add(url)

        try:
            response = requests.get(url, timeout=5)
            soup = BeautifulSoup(response.text, 'html.parser')

            for link in soup.find_all('a', href=True):
                full_link = urljoin(url, link['href'])
                parsed = urlparse(full_link)

                
                if parsed.netloc != base_domain:
                    continue

                
                if any(x in full_link for x in [
                    "cart", "login", "wishlist", "translate",
                    "account", "checkout", "search"
                ]):
                    continue

                path = parsed.path

               
                if path.startswith("/product/"):
                    handle = path.split("/product/")[1]
                    handle = handle.split("?")[0]  # limpar params
                    handle = handle.strip("/")

                    if handle not in product_handles:
                        product_handles.add(handle)
                        print(f"  Produto encontrado: {handle}")

                    continue 

                
                if full_link not in seen:
                    seen.add(full_link)
                    queue.append((full_link, depth + 1, max_depth))

            print(f"  -> Total produtos: {len(product_handles)}")
            print(f"  -> Links vistos: {len(seen)}")

            time.sleep(1)

        except Exception as e:
            print(f"[ERRO] {url}: {e}")

    return product_handles


variaveis

In [ ]:
start_points = {
        "https://www.inocos.com/": 1,
        "https://www.inocos.com/category": 2,
    }

products = crawl_products(start_points)


[DEPTH 0/1] A visitar: https://www.inocos.com/
  Produto encontrado: mystery-box-inocos
  Produto encontrado: box-verniz-gel-cateye-celestial-inocos
  Produto encontrado: iman-magnetico-inocos-cateye-5-em-1
  Produto encontrado: verniz-gel-inocos-cateye-aurora-boreal
  Produto encontrado: verniz-gel-inocos-cateye-cometa
  Produto encontrado: top-coat-inocos-cafe-natural
  Produto encontrado: multi-portable-nail-drill-inocos
  Produto encontrado: almofada-inocos-manicure-apoio-de-bracos
  Produto encontrado: box-glitter-biodegradavel-bioglitter-inocos
  Produto encontrado: glow-pro-nail-desk-lamp-candeeiro-inocos
  Produto encontrado: kit-complementos-manicure-russa-combinada-inocos
  -> Total produtos: 11
  -> Links vistos: 153
[DEPTH 0/2] A visitar: https://www.inocos.com/category
  -> Total produtos: 11
  -> Links vistos: 153
[DEPTH 1/1] A visitar: https://www.inocos.com/page/pontos-de-venda
  -> Total produtos: 11
  -> Links vistos: 184
[DEPTH 1/1] A visitar: https://www.inocos.com/

In [15]:
print("\n Produtos encontrados:")
for p in products:
    print(p)

print(f"\nTotal: {len(products)} produtos")


 Produtos encontrados:
placa-de-carimbos-inocos-12-xl-lace
cola-para-cristais-inocos
oleo-de-cuticulas-inocos-de-framboesa-30ml
foil-inocos-no3
verniz-gel-inocos-piadas-a-pai
like-gel-inocos-199-rosa-cocktail
placa-de-carimbos-inocos-09-urban
base-inocos-branco-gelo-pb04
glitter-biodegradavel-bioglitter-inocos-06-rosa-claro-04-mm
verniz-gel-inocos-gengibre-natura-lovers-spice-edition
fiber-base-gel-inocos-nude-perola
natura-flowers-inocos
nail-liner-inocos-coral-pastel
oleo-de-cuticulas-inocos-de-algodao-doce-30ml
like-gel-inocos-185
like-gel-inocos-200-rosa-gelado
removedor-de-verniz-gel-inocos-150ml
like-gel-inocos-225-nude-areia-leitoso
perfect-powder-inocos-azul-marinheiro-p53
verniz-gel-inocos-boca-doce
top-coat-aurora-inocos
glitter-po-efeito-2-em-1-inocos-magia-ouro
perfect-powder-inocos-rosa-ferrari-p67
like-gel-inocos-196-laranja-papaia
bond-primer-inocos
verniz-gel-inocos-papoila
verniz-gel-inocos-fingi-que-nao-vi
alicate-unhas-articuled-articulado-pedicure-17-milimetros-mm-

Pré-processamento dos dados


In [22]:
def clean_handle(handle):
    remove_words = {"inocos", "hifans"}
    
    parts = handle.lower().split("-")
    cleaned = [p for p in parts if p not in remove_words]
    
    return " ".join(cleaned)

variaveis


In [23]:
dados_processados = [clean_handle(p) for p in products]
print(dados_processados)

['placa de carimbos 12 xl lace', 'cola para cristais', 'oleo de cuticulas de framboesa 30ml', 'foil no3', 'verniz gel piadas a pai', 'like gel 199 rosa cocktail', 'placa de carimbos 09 urban', 'base branco gelo pb04', 'glitter biodegradavel bioglitter 06 rosa claro 04 mm', 'verniz gel gengibre natura lovers spice edition', 'fiber base gel nude perola', 'natura flowers', 'nail liner coral pastel', 'oleo de cuticulas de algodao doce 30ml', 'like gel 185', 'like gel 200 rosa gelado', 'removedor de verniz gel 150ml', 'like gel 225 nude areia leitoso', 'perfect powder azul marinheiro p53', 'verniz gel boca doce', 'top coat aurora', 'glitter po efeito 2 em 1 magia ouro', 'perfect powder rosa ferrari p67', 'like gel 196 laranja papaia', 'bond primer', 'verniz gel papoila', 'verniz gel fingi que nao vi', 'alicate unhas articuled articulado pedicure 17 milimetros mm np10', 'verniz gel koala', 'builder gel rosa leitoso intenso de media viscosidade 50g', 'cristais d2', 'kit moldes f1', 'madrepero